In [1]:
import pandas as pd

df = pd.read_csv('../../datasets/US_with_labels.csv')
df.head()

#print(df[df["name"] == "APT."])

# print(df['popularity'].value_counts())
# df['popularity'].value_counts().plot(kind='bar')
# plt.title('Popularity Distribution')

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,0,0,US,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.000000,0.2480,0.576,138.008,4,About_Average
1,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,2,0,1,US,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.000000,0.1410,0.214,101.061,4,Higher
2,0aB0v4027ukVziUGwVGYpG,tv off (feat. lefty gunplay),"Kendrick Lamar, Lefty Gunplay",3,0,-1,US,2025-02-17,92,True,...,-6.679,0,0.2630,0.0837,0.000000,0.4230,0.548,100.036,4,Higher
3,3GCdLUSnKSMJhs4Tj6CV3s,All The Stars (with SZA),"Kendrick Lamar, SZA",4,0,12,US,2025-02-17,90,True,...,-4.946,1,0.0599,0.0612,0.000195,0.0926,0.557,96.782,4,About_Average
4,0nj9Bq5sHDiTxSHunhgkFb,squabble up,Kendrick Lamar,5,0,4,US,2025-02-17,89,True,...,-5.568,1,0.1980,0.0206,0.000000,0.0783,0.711,103.921,4,Lower


In [2]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import RidgeClassifier

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [3]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RidgeClassifier()
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("Ridge Regression")
mean_squared_error(y_test, y_pred)


Ridge Regression


257.4457908163265

In [4]:
pred = pd.DataFrame(y_pred).value_counts()
test = pd.DataFrame(y_test).value_counts()

print(pred.describe())
print(test.describe())

count     21.000000
mean     224.000000
std      292.645349
min        1.000000
25%       16.000000
50%       77.000000
75%      356.000000
max      958.000000
Name: count, dtype: float64
count     73.000000
mean      64.438356
std      105.525666
min        1.000000
25%        2.000000
50%        9.000000
75%       77.000000
max      408.000000
Name: count, dtype: float64


In [5]:
hyperParameters = {'ridgeclassifier__alpha':[150000, 200000, 250000, 300000, 350000, 400000, 450000, 500000, 550000, 600000, 650000, 700000, 750000, 800000, 850000, 900000, 950000]}

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

RRGrid = GridSearchCV(
    pipeline,
    param_grid=hyperParameters,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    verbose=3,
)

RRGrid.fit(X_train, y_train)
print("Best alpha: ", RRGrid.best_params_["ridgeclassifier__alpha"])
print("Best score: ", RRGrid.best_score_)

Fitting 5 folds for each of 17 candidates, totalling 85 fits
[CV 1/5] END ..ridgeclassifier__alpha=150000;, score=-136.238 total time=   0.1s
[CV 2/5] END ..ridgeclassifier__alpha=150000;, score=-160.703 total time=   0.2s
[CV 3/5] END ..ridgeclassifier__alpha=150000;, score=-138.810 total time=   0.2s
[CV 4/5] END ..ridgeclassifier__alpha=150000;, score=-157.643 total time=   0.2s
[CV 5/5] END ..ridgeclassifier__alpha=150000;, score=-157.223 total time=   0.1s
[CV 1/5] END ..ridgeclassifier__alpha=200000;, score=-136.171 total time=   0.2s
[CV 2/5] END ..ridgeclassifier__alpha=200000;, score=-160.932 total time=   0.1s
[CV 3/5] END ..ridgeclassifier__alpha=200000;, score=-138.189 total time=   0.1s
[CV 4/5] END ..ridgeclassifier__alpha=200000;, score=-157.643 total time=   0.2s
[CV 5/5] END ..ridgeclassifier__alpha=200000;, score=-157.223 total time=   0.1s
[CV 1/5] END ..ridgeclassifier__alpha=250000;, score=-136.171 total time=   0.1s
[CV 2/5] END ..ridgeclassifier__alpha=250000;, s